In [0]:
print("Sql-Server database migration")

In [0]:
jdbc_url = "jdbc:sqlserver://rainbow-server.database.windows.net:1433;database=rainbow-db"

connection_properties = {
    "user": dbutils.secrets.get(scope="sc-rainbow-vault-batch02", key="sql-server-user"),
    "password": dbutils.secrets.get(scope="sc-rainbow-vault-batch02", key="sql-server-password"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
info_query = "(select TABLE_SCHEMA, TABLE_NAME from information_schema.tables where TABLE_TYPE = 'BASE TABLE') as src"

In [0]:
tables_info_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", info_query)
    .option("user", connection_properties["user"])
    .option("password", connection_properties["password"])
    .load()
)

display(tables_info_df)

In [0]:
for table_info in tables_info_df.collect():
    schema = table_info.TABLE_SCHEMA
    table_name = table_info.TABLE_NAME
    print(f"Table {schema}.{table_name} exists")

    query = f"(select * from {schema}.{table_name}) as src"

    try:
        df = (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", query)
            .option("user", connection_properties["user"])
            .option("password", connection_properties["password"])
            .load()
        )

        target_file_name = table_name.strip().replace(" ", "")
        print(f"saving the table into - {target_file_name}")

        (
            df.write.format("csv")
            .mode("overwrite")
            .save(f"/mnt/rainbowcontainer/sql_server_data/{target_file_name}")
        )

    except Exception as e:
        print(f"Error - {e}")
    